# Train/Test Split Visualization

Compare the distribution of training and testing sets created by:
1. **Random split** (shuffled 80/20, seed=42)
2. **Kennard-Stone split** (maximizes chemical diversity in training set)

Visualizations:
- PCA projection of fingerprint space (train vs test)
- pChEMBL value distributions (train vs test)
- t-SNE embedding colored by split assignment

## 1. Setup & Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/dataprofessor/aromatase.git'
REPO_DIR = '/content/aromatase'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

SPLITS_DIR = os.path.join(REPO_DIR, 'data', 'splits')
CURATED_PATH = os.path.join(REPO_DIR, 'data', 'processed', 'aromatase_bioactivity_curated.csv')
print(f'Splits dir: {SPLITS_DIR}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

## 2. Load Data

In [ ]:
# Load target values
curated = pd.read_csv(CURATED_PATH, usecols=['molecule_chembl_id', 'pchembl_value'])
curated = curated.dropna(subset=['pchembl_value']).drop_duplicates('molecule_chembl_id')
target_map = curated.set_index('molecule_chembl_id')['pchembl_value'].to_dict()

# Choose a fingerprint for visualization (MACCS is compact and interpretable)
FP_NAME = 'maccs'  # Change to any: pubchem, cdk, klekota_roth, etc.

def load_split(fp_name, split_name, subset):
    """Load a split file and add pchembl values."""
    path = os.path.join(SPLITS_DIR, f'aromatase_{fp_name}_fp_{split_name}_{subset}.csv')
    df = pd.read_csv(path)
    fp_cols = [c for c in df.columns if c != 'molecule_chembl_id']
    df['pchembl_value'] = df['molecule_chembl_id'].map(target_map)
    df = df.dropna(subset=['pchembl_value'])
    return df, fp_cols

# Load all 4 sets
train_rand, fp_cols = load_split(FP_NAME, 'random', 'train')
test_rand, _ = load_split(FP_NAME, 'random', 'test')
train_ks, _ = load_split(FP_NAME, 'kennard_stone', 'train')
test_ks, _ = load_split(FP_NAME, 'kennard_stone', 'test')

print(f'Fingerprint: {FP_NAME.upper()} ({len(fp_cols)} features)')
print(f'Random split:        train={len(train_rand)}, test={len(test_rand)}')
print(f'Kennard-Stone split: train={len(train_ks)}, test={len(test_ks)}')

## 3. pChEMBL Value Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Random split
axes[0].hist(train_rand['pchembl_value'], bins=30, alpha=0.7, label='Train', color='steelblue', density=True)
axes[0].hist(test_rand['pchembl_value'], bins=30, alpha=0.7, label='Test', color='coral', density=True)
axes[0].set_xlabel('pChEMBL value')
axes[0].set_ylabel('Density')
axes[0].set_title('Random Split')
axes[0].legend()
axes[0].axvline(train_rand['pchembl_value'].mean(), color='steelblue', ls='--', lw=1)
axes[0].axvline(test_rand['pchembl_value'].mean(), color='coral', ls='--', lw=1)

# Kennard-Stone split
axes[1].hist(train_ks['pchembl_value'], bins=30, alpha=0.7, label='Train', color='steelblue', density=True)
axes[1].hist(test_ks['pchembl_value'], bins=30, alpha=0.7, label='Test', color='coral', density=True)
axes[1].set_xlabel('pChEMBL value')
axes[1].set_title('Kennard-Stone Split')
axes[1].legend()
axes[1].axvline(train_ks['pchembl_value'].mean(), color='steelblue', ls='--', lw=1)
axes[1].axvline(test_ks['pchembl_value'].mean(), color='coral', ls='--', lw=1)

plt.suptitle(f'pChEMBL Distribution: Train vs Test ({FP_NAME.upper()} fingerprint)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Summary stats
print(f'\nRandom split  — Train mean: {train_rand["pchembl_value"].mean():.3f}, Test mean: {test_rand["pchembl_value"].mean():.3f}')
print(f'Kennard-Stone — Train mean: {train_ks["pchembl_value"].mean():.3f}, Test mean: {test_ks["pchembl_value"].mean():.3f}')

## 4. PCA Projection (Chemical Space)

In [ ]:
# Combine all data for a single PCA fit
all_mols = pd.concat([train_rand, test_rand], ignore_index=True).drop_duplicates('molecule_chembl_id')
X_all = all_mols[fp_cols].values.astype(np.float32)

# Fit PCA on all molecules
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all)
all_mols['PC1'] = X_pca[:, 0]
all_mols['PC2'] = X_pca[:, 1]

# Map PCA coordinates back to each split
pca_lookup = all_mols.set_index('molecule_chembl_id')[['PC1', 'PC2']]

def get_pca_coords(df):
    return df[['molecule_chembl_id']].merge(pca_lookup, on='molecule_chembl_id', how='left')

train_rand_pca = get_pca_coords(train_rand)
test_rand_pca = get_pca_coords(test_rand)
train_ks_pca = get_pca_coords(train_ks)
test_ks_pca = get_pca_coords(test_ks)

print(f'PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Random split PCA
axes[0].scatter(train_rand_pca['PC1'], train_rand_pca['PC2'],
               alpha=0.3, s=10, c='steelblue', label='Train')
axes[0].scatter(test_rand_pca['PC1'], test_rand_pca['PC2'],
               alpha=0.5, s=15, c='coral', label='Test')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[0].set_title('Random Split')
axes[0].legend()

# Kennard-Stone PCA
axes[1].scatter(train_ks_pca['PC1'], train_ks_pca['PC2'],
               alpha=0.3, s=10, c='steelblue', label='Train')
axes[1].scatter(test_ks_pca['PC1'], test_ks_pca['PC2'],
               alpha=0.5, s=15, c='coral', label='Test')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].set_title('Kennard-Stone Split')
axes[1].legend()

plt.suptitle(f'PCA of Chemical Space: Train vs Test ({FP_NAME.upper()})', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. t-SNE Embedding

In [ ]:
# t-SNE on all molecules (takes ~30s for 3K molecules)
print('Running t-SNE (this may take ~30 seconds)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_all)
all_mols['tSNE1'] = X_tsne[:, 0]
all_mols['tSNE2'] = X_tsne[:, 1]

tsne_lookup = all_mols.set_index('molecule_chembl_id')[['tSNE1', 'tSNE2']]

def get_tsne_coords(df):
    return df[['molecule_chembl_id']].merge(tsne_lookup, on='molecule_chembl_id', how='left')

train_rand_tsne = get_tsne_coords(train_rand)
test_rand_tsne = get_tsne_coords(test_rand)
train_ks_tsne = get_tsne_coords(train_ks)
test_ks_tsne = get_tsne_coords(test_ks)
print('Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Random split t-SNE
axes[0].scatter(train_rand_tsne['tSNE1'], train_rand_tsne['tSNE2'],
               alpha=0.3, s=10, c='steelblue', label='Train')
axes[0].scatter(test_rand_tsne['tSNE1'], test_rand_tsne['tSNE2'],
               alpha=0.5, s=15, c='coral', label='Test')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')
axes[0].set_title('Random Split')
axes[0].legend()

# Kennard-Stone t-SNE
axes[1].scatter(train_ks_tsne['tSNE1'], train_ks_tsne['tSNE2'],
               alpha=0.3, s=10, c='steelblue', label='Train')
axes[1].scatter(test_ks_tsne['tSNE1'], test_ks_tsne['tSNE2'],
               alpha=0.5, s=15, c='coral', label='Test')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].set_title('Kennard-Stone Split')
axes[1].legend()

plt.suptitle(f't-SNE of Chemical Space: Train vs Test ({FP_NAME.upper()})', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
from scipy.stats import ks_2samp

# Kolmogorov-Smirnov test: are train and test distributions different?
ks_rand = ks_2samp(train_rand['pchembl_value'], test_rand['pchembl_value'])
ks_ks = ks_2samp(train_ks['pchembl_value'], test_ks['pchembl_value'])

print('=== Distribution Comparison (KS test on pChEMBL) ===')
print(f'Random split:        KS stat={ks_rand.statistic:.4f}, p={ks_rand.pvalue:.4f}')
print(f'Kennard-Stone split: KS stat={ks_ks.statistic:.4f}, p={ks_ks.pvalue:.4f}')
print()
print('Interpretation:')
print('  - Random split: train/test should have similar distributions (high p-value)')
print('  - Kennard-Stone: train covers more chemical space, test may differ more')

# Feature space coverage
print(f'\n=== Feature Space Coverage ===')
print(f'Random split  — Train PC1 range: [{train_rand_pca["PC1"].min():.2f}, {train_rand_pca["PC1"].max():.2f}]')
print(f'                Test PC1 range:  [{test_rand_pca["PC1"].min():.2f}, {test_rand_pca["PC1"].max():.2f}]')
print(f'Kennard-Stone — Train PC1 range: [{train_ks_pca["PC1"].min():.2f}, {train_ks_pca["PC1"].max():.2f}]')
print(f'                Test PC1 range:  [{test_ks_pca["PC1"].min():.2f}, {test_ks_pca["PC1"].max():.2f}]')